# Option Metrics Calculator

This notebook loads **existing option quotes** and computes practical trading metrics such as moneyness, break-even, intrinsic/extrinsic value, annualized risk measures, probability proxy, and Black-Scholes Greeks.

## Expected input columns

Provide a CSV (or DataFrame) with at least the following columns:

- `symbol`
- `option_type` (`call` or `put`)
- `strike`
- `expiry` (YYYY-MM-DD)
- `underlying_price`
- `option_price` (mid/last/theoretical, your choice)
- `implied_vol` (decimal form, e.g. 0.25)

Optional columns:
- `risk_free_rate` (decimal; defaults to 0.04 if omitted)
- `contracts` (position size; defaults to 1)

In [1]:
import math
from datetime import datetime, timezone

import numpy as np
import pandas as pd

In [27]:
# --- Configure your data source ---
CSV_PATH = 'C:\\Users\\alezu\\Documents\\Projects\\option-risk\\data\\option_data_yf_MSFT_20251106.csv'  # Update this path to your option quote file
DEFAULT_RF = 0.04  # Fallback risk-free rate when not provided

In [32]:
df = pd.read_csv(CSV_PATH)
df['symbol'] = "MSFT"
df['option_type'] = df['option_type'].apply(lambda x: 'call' if x else 'put')
df.head()

,strike,implied_vol,option_price,bid,ask,expiry,inTheMoney,option_type,days_to_expiry,T,expiry.1,disc_factor,div_yield,forward,underlying_price,symbol
0,305.0,2.469608,207.50,208.15,209.4,2025-11-21,True,call,14,0.038356,2025-11-21,0.998501,0.007,496.221801,497.100006,MSFT
1,502.5,0.015635,7.55,0.00,0.0,2025-11-21,False,call,14,0.038356,2025-11-21,0.998501,0.007,496.221801,497.100006,MSFT
2,505.0,0.015635,6.59,0.00,0.0,2025-11-21,False,call,14,0.038356,2025-11-21,0.998501,0.007,496.221801,497.100006,MSFT
3,507.5,0.031260,5.65,0.00,0.0,2025-11-21,False,call,14,0.038356,2025-11-21,0.998501,0.007,496.221801,497.100006,MSFT
4,510.0,0.031260,4.85,0.00,0.0,2025-11-21,False,call,14,0.038356,2025-11-21,0.998501,0.007,496.221801,497.100006,MSFT


In [33]:
required_cols = {
    'symbol', 'option_type', 'strike', 'expiry', 'underlying_price',
    'option_price', 'implied_vol'
}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f'Missing required columns: {sorted(missing)}')

df = df.copy()
df['option_type'] = df['option_type'].str.lower().str.strip()
df['expiry'] = pd.to_datetime(df['expiry'], utc=True, errors='coerce')
if df['expiry'].isna().any():
    bad_idx = df.index[df['expiry'].isna()].tolist()
    raise ValueError(f'Invalid expiry dates at rows: {bad_idx}')

for col in ['strike', 'underlying_price', 'option_price', 'implied_vol']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

if 'risk_free_rate' not in df.columns:
    df['risk_free_rate'] = DEFAULT_RF
else:
    df['risk_free_rate'] = pd.to_numeric(df['risk_free_rate'], errors='coerce').fillna(DEFAULT_RF)

if 'contracts' not in df.columns:
    df['contracts'] = 1
else:
    df['contracts'] = pd.to_numeric(df['contracts'], errors='coerce').fillna(1).clip(lower=0)

if df[['strike', 'underlying_price', 'option_price', 'implied_vol']].isna().any().any():
    raise ValueError('Numeric columns contain invalid values after parsing.')

df.head()

,strike,implied_vol,option_price,bid,ask,expiry,inTheMoney,option_type,days_to_expiry,T,expiry.1,disc_factor,div_yield,forward,underlying_price,symbol,risk_free_rate,contracts
0,305.0,2.469608,207.50,208.15,209.4,2025-11-21 00:00:00+00:00,True,call,14,0.038356,2025-11-21,0.998501,0.007,496.221801,497.100006,MSFT,0.04,1
1,502.5,0.015635,7.55,0.00,0.0,2025-11-21 00:00:00+00:00,False,call,14,0.038356,2025-11-21,0.998501,0.007,496.221801,497.100006,MSFT,0.04,1
2,505.0,0.015635,6.59,0.00,0.0,2025-11-21 00:00:00+00:00,False,call,14,0.038356,2025-11-21,0.998501,0.007,496.221801,497.100006,MSFT,0.04,1
3,507.5,0.031260,5.65,0.00,0.0,2025-11-21 00:00:00+00:00,False,call,14,0.038356,2025-11-21,0.998501,0.007,496.221801,497.100006,MSFT,0.04,1
4,510.0,0.031260,4.85,0.00,0.0,2025-11-21 00:00:00+00:00,False,call,14,0.038356,2025-11-21,0.998501,0.007,496.221801,497.100006,MSFT,0.04,1


In [34]:
def norm_cdf(x: float) -> float:
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))


def norm_pdf(x: float) -> float:
    return math.exp(-0.5 * x * x) / math.sqrt(2.0 * math.pi)


def bs_greeks(option_type: str, s: float, k: float, t: float, r: float, sigma: float):
    # Handle near-zero time or vol to avoid unstable division
    t = max(t, 1e-8)
    sigma = max(sigma, 1e-8)

    d1 = (math.log(s / k) + (r + 0.5 * sigma**2) * t) / (sigma * math.sqrt(t))
    d2 = d1 - sigma * math.sqrt(t)

    if option_type == 'call':
        delta = norm_cdf(d1)
        theta = (
            -(s * norm_pdf(d1) * sigma) / (2 * math.sqrt(t))
            - r * k * math.exp(-r * t) * norm_cdf(d2)
        )
        rho = k * t * math.exp(-r * t) * norm_cdf(d2)
    elif option_type == 'put':
        delta = norm_cdf(d1) - 1
        theta = (
            -(s * norm_pdf(d1) * sigma) / (2 * math.sqrt(t))
            + r * k * math.exp(-r * t) * norm_cdf(-d2)
        )
        rho = -k * t * math.exp(-r * t) * norm_cdf(-d2)
    else:
        raise ValueError(f'Unknown option_type: {option_type}')

    gamma = norm_pdf(d1) / (s * sigma * math.sqrt(t))
    vega = s * norm_pdf(d1) * math.sqrt(t)

    # theta returned in annualized terms; trading desks often use theta/day
    theta_per_day = theta / 365.0

    return {
        'd1': d1,
        'd2': d2,
        'delta': delta,
        'gamma': gamma,
        'vega': vega / 100.0,      # per +1 vol point
        'theta_day': theta_per_day,
        'rho': rho / 100.0         # per +1% rate move
    }

In [35]:
now = pd.Timestamp(datetime.now(timezone.utc))
df['days_to_expiry'] = (df['expiry'] - now).dt.total_seconds() / 86400.0
df['days_to_expiry'] = df['days_to_expiry'].clip(lower=0)
df['t_years'] = (df['days_to_expiry'] / 365.0).clip(lower=1e-8)

is_call = df['option_type'] == 'call'
is_put = df['option_type'] == 'put'

if (~(is_call | is_put)).any():
    bad_types = df.loc[~(is_call | is_put), 'option_type'].unique().tolist()
    raise ValueError(f'option_type must be call/put only. Found: {bad_types}')

# Intrinsic / Extrinsic
df['intrinsic'] = np.where(is_call,
                           np.maximum(df['underlying_price'] - df['strike'], 0.0),
                           np.maximum(df['strike'] - df['underlying_price'], 0.0))
df['extrinsic'] = df['option_price'] - df['intrinsic']

# Moneyness & break-even
df['moneyness_spot_over_strike'] = df['underlying_price'] / df['strike']
df['break_even'] = np.where(is_call,
                            df['strike'] + df['option_price'],
                            df['strike'] - df['option_price'])

# Exposure
df['max_loss_long'] = df['option_price'] * 100 * df['contracts']
df['notional_underlying'] = df['underlying_price'] * 100 * df['contracts']
df['premium_pct_of_underlying'] = (df['option_price'] / df['underlying_price']) * 100

# N(d2) style probability proxy (risk-neutral ITM probability under BS assumptions)
probs = []
greek_rows = []
for row in df.itertuples(index=False):
    greeks = bs_greeks(
        option_type=row.option_type,
        s=float(row.underlying_price),
        k=float(row.strike),
        t=float(row.t_years),
        r=float(row.risk_free_rate),
        sigma=float(row.implied_vol),
    )
    greek_rows.append(greeks)
    probs.append(norm_cdf(greeks['d2']) if row.option_type == 'call' else norm_cdf(-greeks['d2']))

greeks_df = pd.DataFrame(greek_rows)
df = pd.concat([df.reset_index(drop=True), greeks_df.reset_index(drop=True)], axis=1)
df['prob_itm_proxy'] = probs

# Position Greeks (per contract multiplier = 100)
for g in ['delta', 'gamma', 'vega', 'theta_day', 'rho']:
    df[f'position_{g}'] = df[g] * 100 * df['contracts']

metrics_cols = [
    'symbol', 'option_type', 'expiry', 'strike', 'underlying_price', 'option_price',
    'days_to_expiry', 'implied_vol', 'intrinsic', 'extrinsic',
    'moneyness_spot_over_strike', 'break_even', 'premium_pct_of_underlying',
    'prob_itm_proxy', 'delta', 'gamma', 'vega', 'theta_day', 'rho',
    'position_delta', 'position_gamma', 'position_vega', 'position_theta_day', 'position_rho'
]

result = df[metrics_cols].sort_values(['symbol', 'expiry', 'strike']).reset_index(drop=True)
result.head(20)

,symbol,option_type,expiry,strike,underlying_price,option_price,days_to_expiry,implied_vol,intrinsic,extrinsic,...,delta,gamma,vega,theta_day,rho,position_delta,position_gamma,position_vega,position_theta_day,position_rho
0,MSFT,put,2025-11-21 00:00:00+00:00,260.0,497.100006,0.02,0.0,0.500005,0.000000,0.020000,...,0.0,0.0,0.0,0.000000,-0.000000e+00,0.0,0.0,0.0,0.000000,-0.000000
1,MSFT,put,2025-11-21 00:00:00+00:00,265.0,497.100006,0.02,0.0,0.500005,0.000000,0.020000,...,0.0,0.0,0.0,0.000000,-0.000000e+00,0.0,0.0,0.0,0.000000,-0.000000
2,MSFT,put,2025-11-21 00:00:00+00:00,270.0,497.100006,0.03,0.0,0.500005,0.000000,0.030000,...,0.0,0.0,0.0,0.000000,-0.000000e+00,0.0,0.0,0.0,0.000000,-0.000000
3,MSFT,put,2025-11-21 00:00:00+00:00,275.0,497.100006,0.04,0.0,0.500005,0.000000,0.040000,...,0.0,0.0,0.0,0.000000,-0.000000e+00,0.0,0.0,0.0,0.000000,-0.000000
4,MSFT,put,2025-11-21 00:00:00+00:00,280.0,497.100006,0.04,0.0,0.500005,0.000000,0.040000,...,0.0,0.0,0.0,0.000000,-0.000000e+00,0.0,0.0,0.0,0.000000,-0.000000
5,MSFT,put,2025-11-21 00:00:00+00:00,285.0,497.100006,0.02,0.0,0.500005,0.000000,0.020000,...,0.0,0.0,0.0,0.000000,-0.000000e+00,0.0,0.0,0.0,0.000000,-0.000000
6,MSFT,put,2025-11-21 00:00:00+00:00,290.0,497.100006,0.02,0.0,0.500005,0.000000,0.020000,...,0.0,0.0,0.0,0.000000,-0.000000e+00,0.0,0.0,0.0,0.000000,-0.000000
7,MSFT,put,2025-11-21 00:00:00+00:00,295.0,497.100006,0.03,0.0,0.500005,0.000000,0.030000,...,0.0,0.0,0.0,0.000000,-0.000000e+00,0.0,0.0,0.0,0.000000,-0.000000
8,MSFT,put,2025-11-21 00:00:00+00:00,300.0,497.100006,0.01,0.0,0.500005,0.000000,0.010000,...,0.0,0.0,0.0,0.000000,-0.000000e+00,0.0,0.0,0.0,0.000000,-0.000000
9,MSFT,call,2025-11-21 00:00:00+00:00,305.0,497.100006,207.50,0.0,2.469608,192.100006,15.399994,...,1.0,0.0,0.0,-0.033425,3.050000e-08,100.0,0.0,0.0,-3.342466,0.000003


In [36]:
# Portfolio-level snapshot
portfolio = {
    'total_contracts': float(df['contracts'].sum()),
    'total_premium_at_risk': float(df['max_loss_long'].sum()),
    'net_delta': float(df['position_delta'].sum()),
    'net_gamma': float(df['position_gamma'].sum()),
    'net_vega_per_vol_pt': float(df['position_vega'].sum()),
    'net_theta_per_day': float(df['position_theta_day'].sum()),
    'net_rho_per_1pct_rate': float(df['position_rho'].sum()),
}
pd.Series(portfolio).to_frame('value')

,value
total_contracts,1.384000e+03
total_premium_at_risk,3.297234e+06
net_delta,5.432231e+03
net_gamma,1.926460e+01
net_vega_per_vol_pt,2.559002e+03
net_theta_per_day,-4.278719e+02
net_rho_per_1pct_rate,4.092717e+03


In [ ]:
# Save results for downstream workflows
OUTPUT_PATH = '../data/option_metrics_output.csv'
result.to_csv(OUTPUT_PATH, index=False)
print(f'Wrote metrics to: {OUTPUT_PATH}')